### ADVANCED RAG TECHNIQUES

Let's start by digging into `ingest`:

1. No LangChain, just native for maximum flexibility.
2. Let's use an LLM to divide up chunks in a sensible way.
3. Let's use the best chunk size and encoder from yesterday.
4. Let's also have the LLM re-write chunks in a way that's most useful(`document pre-processing`).

In [3]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from rich import print
import os

In [5]:
# Loading environment variables from .env file
db_name = "preprocessed_db"
collection_name = "docs"
embedding_model = "all-MiniLM-L6-v2"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

load_dotenv(override=True)

groq_api_key = os.getenv("GROQ_API_KEY")
groq_model = os.getenv("GROQ_MODEL")
groq_base_url = os.getenv("GROQ_BASE_URL")

if groq_api_key:
    print(f"Groq API Key exists: {groq_api_key[:4]}, {groq_model[-8:]}, {groq_base_url[-8:]}")
else:
    print("Groq API Key not set.")

Groq API Key exists: gsk_, -oss-20b, penai/v1

In [7]:
openai = OpenAI(base_url=groq_base_url, api_key=groq_api_key)

In [8]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [9]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

#### 3 Steps:
1. Fetch documents from the knowledge base, like LangChain did.
2. Call an LLM to turn documents into chunks.
3. Store the chunks in Chroma.

Let's start with step 1

In [10]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [11]:
documents = fetch_documents()

Loaded 76 documents

Step 2: Chunking the documents

In [12]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [13]:
print(make_prompt(documents[0]))

You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: company
The document has been retrieved from: knowledge-base/company/about.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - 
don't leave anything out.
This document should probably be split into 5 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you 
have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

# About Insurellm

Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in 
need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance 
providers.

The company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm 
(auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, 
Insurellm had reached a peak of 200 employees with 12 offices across the US.

However, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable 
growth. This included consolidating office locations, implementing a remote-first strategy, and streamlining 
operations. As of 2025, Insurellm operates with a lean, highly efficient team of 32 employees who have built a 
portfolio of 32 active contracts spanning all eight product lines. The company maintains its San Francisco 
headquarters along with small satellite offices in key markets including New York, Austin, Chicago, and Denver.

Since the restructuring, Insurellm has continued to innovate, expanding its product suite to eight comprehensive 
platforms. The company added Lifellm (life insurance), Healthllm (health insurance), Bizllm (commercial insurance),
and Claimllm (claims processing) to serve the full spectrum of insurance needs. This strategic expansion has been 
highly successful, with strong adoption across all new products:

- **Bizllm** quickly gained traction with 7 commercial insurance contracts, including regional carriers and 
national commercial groups
- **Claimllm** signed 7 contracts ranging from independent adjusting firms to enterprise claims networks
- **Lifellm** secured 6 life insurance clients from small regional providers to major national carriers
- **Healthllm** won 6 health plan contracts including regional insurers and multi-state healthcare alliances

Combined with continued growth in the original product lines (Carllm, Homellm, Markellm, and Rellm), Insurellm now 
serves clients ranging from regional insurers to global reinsurance partners, demonstrating the company's ability 
to compete across the entire insurance value chain.

Respond with the chunks.

In [14]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [15]:
print(make_messages(documents[0]))

[
    {
        'role': 'user',
        'content': "\nYou take a document and you split the document into overlapping chunks for a 
KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: 
company\nThe document has been retrieved from: knowledge-base/company/about.md\n\nA chatbot will use these chunks 
to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the 
entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 
5 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate;
typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval 
results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether
your chunks should represent the entire document with overlap.\n\nHere is the document:\n\n# About 
Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an 
industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with 
insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product 
portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise 
reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the 
US.\n\nHowever, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and 
sustainable growth. This included consolidating office locations, implementing a remote-first strategy, and 
streamlining operations. As of 2025, Insurellm operates with a lean, highly efficient team of 32 employees who have
built a portfolio of 32 active contracts spanning all eight product lines. The company maintains its San Francisco 
headquarters along with small satellite offices in key markets including New York, Austin, Chicago, and 
Denver.\n\nSince the restructuring, Insurellm has continued to innovate, expanding its product suite to eight 
comprehensive platforms. The company added Lifellm (life insurance), Healthllm (health insurance), Bizllm 
(commercial insurance), and Claimllm (claims processing) to serve the full spectrum of insurance needs. This 
strategic expansion has been highly successful, with strong adoption across all new products:\n\n- **Bizllm** 
quickly gained traction with 7 commercial insurance contracts, including regional carriers and national commercial 
groups\n- **Claimllm** signed 7 contracts ranging from independent adjusting firms to enterprise claims networks\n-
**Lifellm** secured 6 life insurance clients from small regional providers to major national carriers\n- 
**Healthllm** won 6 health plan contracts including regional insurers and multi-state healthcare 
alliances\n\nCombined with continued growth in the original product lines (Carllm, Homellm, Markellm, and Rellm), 
Insurellm now serves clients ranging from regional insurers to global reinsurance partners, demonstrating the 
company's ability to compete across the entire insurance value chain.\n\nRespond with the chunks.\n"
    }
]

In [32]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [31]:
print(process_document(documents[0]))


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



BadRequestError: litellm.BadRequestError: GroqException - {"error":{"message":"Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for more details.","type":"invalid_request_error","code":"json_validate_failed","failed_generation":""}}


In [33]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [34]:
chunks = create_chunks(documents)

  0%|          | 0/76 [00:00<?, ?it/s]



Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers



InternalServerError: litellm.InternalServerError: InternalServerError: OpenAIException - Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [ ]:
print(len(chunks))

Step 3: Save the embeddings

In [ ]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [ ]:
create_embeddings(chunks)

In [ ]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

#### LET'S BUILD AN ADVANCED RAG

We'll use these techniques:

* Re-ranking - Reorder the rank results.
* Query re-writing.

In [ ]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [ ]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [ ]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [ ]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

In [ ]:
for chunk in chunks:
    print(chunk.page_content[:15]+"...")

In [ ]:
reranked = rerank(question, chunks)

In [ ]:
for chunk in reranked:
    print(chunk.page_content[:15]+"...")

In [ ]:
question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

In [ ]:
reranked = rerank(question, chunks)

In [ ]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

In [ ]:
reranked[0].page_content

In [ ]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [ ]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [ ]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [ ]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [ ]:
rewrite_query("Who won the IIOTY award?", [])

In [ ]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [ ]:
answer_question("Who won the IIOTY award?", [])

In [ ]:
answer_question("Who went to Manchester University?", [])